# TOFOO Relational Emergence — Colab v3

Pinned, clean-runtime launcher. v3 uses strict position-anchored parsers, interface-only preflight, and invalidates any run where a relational call fails the HYP contract.


In [ ]:
!pip -q install -U 'transformers>=4.37' accelerate bitsandbytes requests
import base64, getpass, json, shutil, subprocess, sys, requests, torch
from pathlib import Path
if not torch.cuda.is_available(): raise RuntimeError('Use Runtime -> Change runtime type -> GPU')
print('GPU:', torch.cuda.get_device_name(0))


## Fetch exact pinned harness


In [ ]:
# Remove every stale local relational-emergence module before fetching.
for p in Path('/content').glob('relational_emergence_v*.py'):
    p.unlink()
shutil.rmtree('/content/__pycache__', ignore_errors=True)

try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if not token:
    token = getpass.getpass('GitHub PAT (Valo-Twin Contents: Read): ')
if not token: raise RuntimeError('No GitHub token supplied')

PINNED_REF='6e5f2b6badd2df742ea01b337b57e63661581db2'
EXPECTED={
    'relational_emergence_v2_core.py':'d90ca4bfff1e7eaf0cd9409ac18bb71417ebeefd',
    'relational_emergence_v3.py':'e61a043e18011fa81fb4d02970ba9dc70ba1fd74',
}
headers={'Authorization':f'Bearer {token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
api='https://api.github.com/repos/nsolland/Valo-Twin/contents/experiments/relational-emergence'
for name, expected_sha in EXPECTED.items():
    r=requests.get(f'{api}/{name}',params={'ref':PINNED_REF},headers=headers,timeout=30)
    if r.status_code!=200: raise RuntimeError(f'GitHub fetch failed for {name}: HTTP {r.status_code}: {r.text[:500]}')
    payload=r.json()
    if payload['sha'] != expected_sha: raise RuntimeError(f'Unexpected blob for {name}: {payload["sha"]} != {expected_sha}')
    code_text=base64.b64decode(payload['content']).decode('utf-8')
    Path('/content',name).write_text(code_text)
    print('fetched:',name,payload['sha'],'lines:',len(code_text.splitlines()))
token=None; headers=None

harness=Path('/content/relational_emergence_v3.py')
core=Path('/content/relational_emergence_v2_core.py')
subprocess.run([sys.executable,'-m','py_compile',str(core),str(harness)],check=True)
print('PINNED REF:',PINNED_REF)
print('HARNESS:',harness.name)
print('COMPILE: PASS')


## 0A. Deterministic + adversarial self-check


In [ ]:
self_out='/content/tofoo_v3_selfcheck'
shutil.rmtree(self_out,ignore_errors=True)
p=subprocess.run([sys.executable,str(harness),'--self-test-only','--out',self_out],capture_output=True,text=True)
print(p.stdout)
if p.returncode!=0:
    print(p.stderr)
    raise RuntimeError(f'SELF-CHECK PROCESS FAILED rc={p.returncode}: {p.stderr[-2000:]}')
selfcheck=json.loads(Path(self_out,'manifest.json').read_text())
assert selfcheck['instrument_status']=='SELF_CHECK_PASS',selfcheck
sc=selfcheck['self_check']
assert sc['preflight_scope']=='INTERFACE_ONLY',selfcheck
assert sc['capability_not_instrument_validity']=='PASS',selfcheck
assert sc['runner_binding']=='CORE_PATCHED_DIRECTLY',selfcheck
assert sc['parser_anchor_contract']=='PASS',selfcheck
assert sc['no_operator_from_evidence_id']=='PASS',selfcheck
assert sc['no_heuristic_extra_token_recovery']=='PASS',selfcheck
print('SELF CHECK + ADVERSARIAL PARSER CHECKS: PASS')


## 0B. End-to-end mock check


In [ ]:
mock='/content/tofoo_v3_mock'
shutil.rmtree(mock,ignore_errors=True)
p=subprocess.run([sys.executable,str(harness),'--mock','--worlds','3','--participants','3','--rounds','3','--operators','3','--out',mock],capture_output=True,text=True)
print(p.stdout)
if p.returncode!=0:
    print(p.stderr)
    raise RuntimeError(f'MOCK PROCESS FAILED rc={p.returncode}: {p.stderr[-2000:]}')
m=json.loads(Path(mock,'manifest.json').read_text()); d=m['diagnostics']
assert m['instrument_status']=='VALID_SIGNAL_DISCOVERY_RUN',m
assert d['oracle_dsl']==1.0 and d['oracle_transfer']==1.0,d
assert d['relational']==1.0 and d['relational_transfer']==1.0,d
assert d['direct_parse_coverage']==1.0,d
assert d['relational_call_parse_coverage']==1.0,d
print('MOCK E2E: PASS')


## 1. Real Qwen2.5-1.5B run

If any relational call produces no structurally valid HYP record, the harness rewrites the result to TEST_INVALID_RELATIONAL_INTERFACE. Do not interpret such a run scientifically.


In [ ]:
out='/content/tofoo_relational_results_v3'
shutil.rmtree(out,ignore_errors=True)
p=subprocess.run([sys.executable,str(harness),'--worlds','3','--participants','3','--rounds','3','--operators','3','--model-a','Qwen/Qwen2.5-1.5B-Instruct','--out',out],capture_output=True,text=True)
print(p.stdout)
if p.stderr: print(p.stderr)
manifest=json.loads(Path(out,'manifest.json').read_text())
print(json.dumps(manifest,indent=2))
if p.returncode!=0: raise RuntimeError(f"{manifest.get('instrument_status')}: {manifest.get('error','see diagnostics')}")


## 2. Inspect result


In [ ]:
for name in ['summary.json','fields.json','raw_calls.json']:
    pth=Path(out,name)
    print('\n###',name)
    print(json.dumps(json.loads(pth.read_text()),indent=2)[:16000])
